# SWAP-Stress: Unified Training Table (obs + EE covariates)

This notebook demonstrates how SWAP-Stress combines:
- standardized observations (`theta`, `suction_cm`, `depth_cm`, `rosetta_level`)
- Earth Engine feature tables

into a unified **observation-level** training table (target: `log10_suction_cm`).

We emphasize:
- schema + join health
- distributions by source
- feature-group summaries (Landsat/Sentinel-1/SMAP/etc.)
- maps at CONUS and global scales (when coordinates exist)

By design, this notebook imports and uses existing code in:
- `map/data/build_training_table.py`
- `map/data/features.py`

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = os.environ.get(
    "SWAPSTRESS_DATA_ROOT",
    os.path.expanduser("~/data/IrrigationGIS/soils"),
)
DATA_ROOT = str(Path(DATA_ROOT).expanduser())

OUT_DIR = Path("notebooks/_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Guard heavy build step
RUN_BUILD_TRAINING_TABLE = False

# Prefer an explicit path if provided
UNIFIED_OBS_TABLE = os.environ.get("SWAPSTRESS_OBS_TABLE")

print("DATA_ROOT:", DATA_ROOT)
print("UNIFIED_OBS_TABLE env:", UNIFIED_OBS_TABLE)
print("RUN_BUILD_TRAINING_TABLE:", RUN_BUILD_TRAINING_TABLE)

## 1) Load a unified observation-level table

Fast path:
- load an existing parquet from `DATA_ROOT/swapstress/training/`

Slow path (guarded):
- build via `map.data.build_training_table.build_unified_table`

In [ ]:
from glob import glob


def discover_default_table(data_root: str) -> str | None:
    d = Path(data_root) / "swapstress" / "training"
    if not d.exists():
        return None
    # Prefer observation-level tables built by build_training_table.py main
    candidates = sorted(glob(str(d / "obs_level_training*_250m.parquet")))
    if candidates:
        return candidates[0]
    # Fallback: any parquet
    any_pq = sorted(glob(str(d / "*.parquet")))
    return any_pq[0] if any_pq else None


table_path = UNIFIED_OBS_TABLE or discover_default_table(DATA_ROOT)
print("table_path:", table_path)

df = pd.DataFrame()
if table_path and Path(table_path).exists():
    df = pd.read_parquet(table_path)
    print("Loaded:", table_path)
    print("Rows:", len(df), "Cols:", df.shape[1])
elif RUN_BUILD_TRAINING_TABLE:
    from map.data.build_training_table import build_unified_table

    out_dir = Path(DATA_ROOT) / "swapstress" / "training"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "obs_level_training_emb_250m.parquet"

    df = build_unified_table(
        sources=["gshp", "ncss", "mt_mesonet", "reesh"],
        data_root=DATA_ROOT,
        output_path=str(out_path),
        fit_method="bayes",
        include_embeddings=True,
        prefer_preprocessed=True,
    )
else:
    print("No table found. Set SWAPSTRESS_OBS_TABLE or mount data under DATA_ROOT.")

df.head()

## 2) Schema + join health

We summarize:
- observation counts by `source`
- unique `sample_id`
- depth / `rosetta_level` coverage
- basic missingness diagnostics

In [ ]:
if df.empty:
    pass
else:
    # core columns expected from build_training_table
    core = [
        c
        for c in [
            "source",
            "sample_id",
            "theta",
            "log10_suction_cm",
            "depth_cm",
            "rosetta_level",
        ]
        if c in df.columns
    ]
    print("Core columns present:", core)

    if "source" in df.columns:
        print("\nCounts by source:")
        print(df["source"].value_counts())

    if "sample_id" in df.columns:
        print("\nUnique sample_id:", df["sample_id"].nunique())

    if "rosetta_level" in df.columns:
        print("\nCounts by rosetta_level:")
        print(df["rosetta_level"].value_counts().sort_index())

    # Missingness: fraction of rows with any NaN in the feature block
    feature_cols = [
        c
        for c in df.columns
        if c
        not in {
            "source",
            "sample_id",
            "theta",
            "log10_suction_cm",
            "depth_cm",
            "rosetta_level",
        }
    ]
    if feature_cols:
        frac_any_nan = float(df[feature_cols].isna().any(axis=1).mean())
        print("\nRows with any NaN in feature columns:", f"{frac_any_nan:.3%}")
        frac_all_nan = float(df[feature_cols].isna().all(axis=1).mean())
        print("Rows with all-NaN features:", f"{frac_all_nan:.3%}")

### 2a) Feature completeness by source

For each source, we compute the average fraction of non-null feature columns per row.

This is a quick check for join/key problems (e.g., missing EE features for a subset of sites).

In [ ]:
if df.empty:
    pass
else:
    base_cols = {
        "source",
        "sample_id",
        "theta",
        "log10_suction_cm",
        "depth_cm",
        "rosetta_level",
    }
    feat_cols = [c for c in df.columns if c not in base_cols]
    if "source" not in df.columns or not feat_cols:
        print("Missing source and/or feature columns")
    else:
        # Per-row completeness ratio (sample for very large tables)
        dd = df
        if len(dd) > 1_000_000:
            dd = dd.sample(n=1_000_000, random_state=7)
        completeness = dd[feat_cols].notna().mean(axis=1)
        out = (
            pd.DataFrame({"source": dd["source"], "feature_completeness": completeness})
            .groupby("source")["feature_completeness"]
            .agg(["count", "mean", "std"])
            .sort_values("mean")
        )
        out

## 3) Distributions by source

We plot distributions for:
- `theta`
- `log10_suction_cm`
- `depth_cm`

Note: for very large tables, we sample rows for plotting.

In [ ]:
if df.empty:
    pass
else:
    d = df.copy()
    plot_cols = [c for c in ["theta", "log10_suction_cm", "depth_cm"] if c in d.columns]
    if "source" not in d.columns or not plot_cols:
        print("Missing required columns for plotting")
    else:
        if len(d) > 300_000:
            d = d.sample(n=300_000, random_state=7)

        if sns is not None:
            sns.set_theme(style="whitegrid")

        fig, axes = plt.subplots(
            1, len(plot_cols), figsize=(5 * len(plot_cols), 4), dpi=150
        )
        axes = np.atleast_1d(axes)
        for ax, col in zip(axes, plot_cols):
            for src, g in d.groupby("source"):
                vals = pd.to_numeric(g[col], errors="coerce").dropna().values
                if len(vals) == 0:
                    continue
                ax.hist(vals, bins=60, alpha=0.4, density=True, label=src)
            ax.set_title(col)
            ax.legend(fontsize=8)
        fig.tight_layout()
        out = OUT_DIR / "training_table_distributions.png"
        fig.savefig(out, dpi=200)
        plt.show()
        print("Saved:", out)

## 4) Feature groups (counts + representative histograms)

We use `map/data/features.py` to classify columns into groups (Landsat, Sentinel-1, SMAP, etc.).

In [ ]:
if df.empty:
    pass
else:
    from map.data.features import get_feature_columns, classify_feature

    feats = get_feature_columns(df, include_depth=False, include_embeddings=True)
    groups = pd.Series([classify_feature(c) for c in feats]).value_counts()
    groups

In [ ]:
if df.empty:
    pass
else:
    from map.data.features import get_feature_columns, classify_feature

    feats = get_feature_columns(df, include_depth=False, include_embeddings=False)

    # Pick a few representative features if present
    wanted = [
        "nd_mean_gs",  # landsat index
        "VV_mean",  # sentinel-1
        "sm_profile_mean",  # smap
        "elevation",  # terrain
    ]
    present = [w for w in wanted if w in df.columns]
    if not present:
        # fallback: pick the first feature in a few groups
        by_group = {}
        for c in feats:
            g = classify_feature(c)
            by_group.setdefault(g, []).append(c)
        for g in ["landsat_indices", "sentinel1", "smap", "terrain"]:
            if g in by_group and by_group[g]:
                present.append(by_group[g][0])

    if not present:
        print("No feature columns found for representative plots")
    else:
        d = df[present + (["source"] if "source" in df.columns else [])].copy()
        if len(d) > 200_000:
            d = d.sample(n=200_000, random_state=7)

        fig, axes = plt.subplots(
            1, len(present), figsize=(5 * len(present), 4), dpi=150
        )
        axes = np.atleast_1d(axes)

        for ax, col in zip(axes, present):
            vals = pd.to_numeric(d[col], errors="coerce")
            # common sentinel from EE export/unmask
            vals = vals.where(vals > -9990)
            ax.hist(vals.dropna().values, bins=60, alpha=0.8)
            ax.set_title(col)

        fig.tight_layout()
        out = OUT_DIR / "training_table_feature_examples.png"
        fig.savefig(out, dpi=200)
        plt.show()
        print("Saved:", out)

## 5) Maps (CONUS + global) if coordinates exist

If the table contains lat/lon columns, we plot quick scatter maps.

Note: this is a lightweight fallback; a publication-style map already exists in `poster/training_data_map.py`.

In [ ]:
if df.empty:
    pass
else:
    lon_col = None
    lat_col = None
    for lc in ["lon", "longitude", "Longitude"]:
        if lc in df.columns:
            lon_col = lc
            break
    for lc in ["lat", "latitude", "Latitude"]:
        if lc in df.columns:
            lat_col = lc
            break

    if lon_col is None or lat_col is None:
        print("No lat/lon columns found")
    else:
        cols = [lon_col, lat_col]
        if "sample_id" in df.columns:
            cols = ["sample_id"] + cols
        pts = df[cols].drop_duplicates().copy()
        pts[lon_col] = pd.to_numeric(pts[lon_col], errors="coerce")
        pts[lat_col] = pd.to_numeric(pts[lat_col], errors="coerce")
        pts = pts.dropna()
        if len(pts) > 200_000:
            pts = pts.sample(n=200_000, random_state=7)

        fig, (ax_g, ax_c) = plt.subplots(2, 1, figsize=(11, 9), dpi=150)
        ax_g.scatter(pts[lon_col], pts[lat_col], s=2, alpha=0.25)
        ax_g.set_title("Global (points only)")
        ax_g.set_xlim(-180, 180)
        ax_g.set_ylim(-60, 85)
        ax_g.set_xlabel("lon")
        ax_g.set_ylabel("lat")

        ax_c.scatter(pts[lon_col], pts[lat_col], s=2, alpha=0.25)
        ax_c.set_title("CONUS (rough extent)")
        ax_c.set_xlim(-130, -65)
        ax_c.set_ylim(24, 51)
        ax_c.set_xlabel("lon")
        ax_c.set_ylabel("lat")

        fig.tight_layout(h_pad=1.5)
        out = OUT_DIR / "training_table_point_maps.png"
        fig.savefig(out, dpi=200)
        plt.show()
        print("Saved:", out)

### 5a) Optional: publication-style map from `poster/training_data_map.py`

A higher-polish global + CONUS map already exists in the repo.

Caution: `poster/training_data_map.py` currently assumes the `~/data/IrrigationGIS/...` directory layout and does not take `DATA_ROOT` as an argument.

In [ ]:
RUN_POSTER_MAP = False

if not RUN_POSTER_MAP:
    print("RUN_POSTER_MAP is False; skipping")
else:
    try:
        from poster.training_data_map import plot_training_data_map

        out_path = OUT_DIR / "poster_training_data_map.png"
        plot_training_data_map(str(out_path), include_rosetta=False)
        print("Saved:", out_path)
    except Exception as e:
        print("Poster map failed:", e)